In [18]:
import setup
setup.init_django()

In [19]:
from blog.models import BlogPost

In [20]:
qs = BlogPost.objects.all().delete()

In [21]:
qs


(4, {'blog.BlogPost': 4})

In [22]:
docs = [
    "The dog jumped over the cat",
    "The cat jumped over the dog",
    "it is very warm today",
    "The cat is yellow and the dog is red"
]


In [23]:
new_data = []
for i, x in enumerate(docs):
    new_data.append(BlogPost(title=f"Blog Post {i+1}", content=x, can_delete=True))

BlogPost.objects.bulk_create(new_data)


[<BlogPost: BlogPost object (9)>,
 <BlogPost: BlogPost object (10)>,
 <BlogPost: BlogPost object (11)>,
 <BlogPost: BlogPost object (12)>]

In [25]:
qs

(4, {'blog.BlogPost': 4})

In [28]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/home/mousavi-m/Documents/talk-to-django/all-MiniLM-L6-v2")

No sentence-transformers model found with name /home/mousavi-m/Documents/talk-to-django/all-MiniLM-L6-v2. Creating a new one with mean pooling.


In [29]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [50]:
def get_embedding(text):
    embedding = model.encode([text])
    return embedding[0]

In [51]:
t = get_embedding("The dog jumped  the cat")

In [55]:
qs = BlogPost.objects.filter(can_delete=True)
for post in qs:
    post.embedding = get_embedding(post.content)
    post.save() 

In [67]:
query = "The dog jumped over the cat"
query_embedding = get_embedding(query)

In [70]:
from pgvector.django import CosineDistance
from django.db.models import F

qs = BlogPost.objects.annotate(
    distance=CosineDistance('embedding', query_embedding),
    simarity=1 - F('distance')
).order_by('simarity')

for obj in qs:
    print(obj.content, obj.distance, obj.simarity)


it is very warm today 1.027799389600016 -0.027799389600015934
The cat is yellow and the dog is red 0.5003317445605779 0.49966825543942206
The cat jumped over the dog 0.01045568189901247 0.9895443181009875
The dog jumped over the cat 0.0 1.0
